# Pose Data Collection Notebook

This notebook extracts pose landmarks from the workout videos, builds the dataset, computes the neutral reference mean, and saves the results as `dataset.csv` and `neutral_mean.npy`.

In [1]:
import os

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

VIDEO_FOLDER = "omar's workout video"
MODEL_PATH = "pose_landmarker_lite.task"

LABEL_MAP = {
    "omar's workout routine (standing)": "neutral",
    "omar's workout routine (left)": "lean_left",
    "omar's workout routine (left2)": "lean_left",
    "omar's workout routine (right)": "lean_right",
    "omar's workout routine (right2)": "lean_right",
    "omar's workout routine (down)": "crouch",
    "omar_s workout routine(Jump)": "jump",
    "omar's workout routine (hands_joined)": "hands_joined",
}

feature_cols = [f"f{i}" for i in range(99)]
rows = []

## Extract landmark rows

Each video is processed frame by frame. When pose landmarks are detected, the 99 landmark values are flattened and stored with the corresponding label.

In [2]:
for fname in sorted(os.listdir(VIDEO_FOLDER)):
    stem = os.path.splitext(fname)[0]
    if stem not in LABEL_MAP:
        print(f"Skipping: {fname}")
        continue

    label = LABEL_MAP[stem]
    path = os.path.join(VIDEO_FOLDER, fname)
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    frame_idx = 0
    clip_start = len(rows)

    base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
    options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
    )
    landmarker = vision.PoseLandmarker.create_from_options(options)

    print(f"Processing '{fname}' -> {label} ...")
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        timestamp_ms = int((frame_idx / fps) * 1000)
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB),
        )
        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.pose_landmarks:
            lm = result.pose_landmarks[0]
            flat = [v for p in lm for v in (p.x, p.y, p.z)]
            rows.append(flat + [label])

        frame_idx += 1

    cap.release()
    landmarker.close()
    print(f"  {frame_idx} frames -> {len(rows) - clip_start} landmark rows extracted")

I0000 00:00:1778440354.486142  277538 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1778440354.489120  277566 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.0.6), renderer: AMD Radeon 780M Graphics (radeonsi, phoenix, ACO, DRM 3.64, 7.0.5-cachyos1.fc44.x86_64)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1778440354.523633  277540 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778440354.539299  277550 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778440354.604705  277542 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


Processing 'omar's workout routine (down).mp4' -> crouch ...
  633 frames -> 633 landmark rows extracted
Processing 'omar's workout routine (hands_joined).mp4' -> hands_joined ...


I0000 00:00:1778440364.146124  277657 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1778440364.147576  277678 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.0.6), renderer: AMD Radeon 780M Graphics (radeonsi, phoenix, ACO, DRM 3.64, 7.0.5-cachyos1.fc44.x86_64)
W0000 00:00:1778440364.175664  277664 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778440364.189003  277660 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  305 frames -> 305 landmark rows extracted
Processing 'omar's workout routine (left).mp4' -> lean_left ...


I0000 00:00:1778440368.628205  277735 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1778440368.629694  277756 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.0.6), renderer: AMD Radeon 780M Graphics (radeonsi, phoenix, ACO, DRM 3.64, 7.0.5-cachyos1.fc44.x86_64)
W0000 00:00:1778440368.658132  277743 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778440368.669561  277747 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


...

  609 frames -> 609 landmark rows extracted
Processing 'omar's workout routine (standing).mp4' -> neutral ...


I0000 00:00:1778440393.128219  278003 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1778440393.129836  278024 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.0.6), renderer: AMD Radeon 780M Graphics (radeonsi, phoenix, ACO, DRM 3.64, 7.0.5-cachyos1.fc44.x86_64)
W0000 00:00:1778440393.164258  278012 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778440393.170426  278011 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  612 frames -> 612 landmark rows extracted
Processing 'omar_s workout routine(Jump).mp4' -> jump ...


I0000 00:00:1778440401.984553  278106 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1778440401.985799  278127 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.0.6), renderer: AMD Radeon 780M Graphics (radeonsi, phoenix, ACO, DRM 3.64, 7.0.5-cachyos1.fc44.x86_64)
W0000 00:00:1778440402.020914  278120 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1778440402.032276  278115 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


  218 frames -> 218 landmark rows extracted


## Build and save the dataset

The neutral class is used as the reference pose. Its mean feature vector is saved separately and subtracted from all rows before exporting the dataset.

In [3]:
df = pd.DataFrame(rows, columns=feature_cols + ["label"])

neutral_rows = df[df["label"] == "neutral"][feature_cols].values
neutral_mean = neutral_rows.mean(axis=0)
np.save("neutral_mean.npy", neutral_mean)
print(f"neutral_mean.npy saved (computed from {len(neutral_rows)} neutral frames)")

df[feature_cols] = df[feature_cols].values - neutral_mean
df.to_csv("dataset.csv", index=False)
print("dataset.csv saved")
print("\nClass distribution:")
print(df["label"].value_counts().to_string())

neutral_mean.npy saved (computed from 612 neutral frames)
dataset.csv saved

Class distribution:
label
lean_left       1087
crouch           633
neutral          612
lean_right       609
hands_joined     305
jump             218
